# 🧬 Bio-JEPA — Pré-entraînement 1000 époques
**Objectif** : Lever la limitation des 100 époques et obtenir un checkpoint plus robuste

---

## ⚙️ Avant de commencer
1. `Runtime → Change runtime type → A100 GPU`
2. Avoir les checkpoints sur Google Drive
3. Durée estimée : **~8-10h** sur A100

| Étape | Description | Durée |
|---|---|---|
| 0 | Setup GPU + Drive + Repo | 5 min |
| 1 | Pré-entraînement 1000 époques | ~8-10h |
| 2 | Évaluation comparative 100ep vs 1000ep | ~30 min |
| 3 | Sauvegarde Drive | 2 min |

---
## Étape 0 — Setup

In [1]:
import torch
!nvidia-smi
print(f'\n✓ GPU : {torch.cuda.get_device_name(0)}')
print(f'✓ VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Wed Mar 11 10:15:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/Bio-JEPA-checkpoints'
DRIVE_RES  = '/content/drive/MyDrive/Bio-JEPA-results'
os.makedirs(DRIVE_RES, exist_ok=True)
print('✓ Drive monté')

Mounted at /content/drive
✓ Drive monté


In [3]:
!pip install torch_geometric rdkit pandas numpy scikit-learn tqdm requests pyyaml chembl-webresource-client -q
print('✓ Dépendances installées')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
✓ Dépendances installées


In [4]:
if not os.path.exists('/content/Bio-JEPA'):
    !git clone https://github.com/7Nayy/Bio-JEPA.git /content/Bio-JEPA
else:
    !git -C /content/Bio-JEPA pull origin main

os.chdir('/content/Bio-JEPA')
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Récupérer le checkpoint 100ep comme référence
shutil.copy(f'{DRIVE_CKPT}/zinc_pretrained.pt', 'checkpoints/zinc_pretrained.pt')
print('✓ Repo prêt')
!ls checkpoints/

Cloning into '/content/Bio-JEPA'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 123 (delta 22), reused 50 (delta 13), pack-reused 54 (from 2)
Receiving objects: 100% (123/123), 48.14 MiB | 34.47 MiB/s, done.
Resolving deltas: 100% (25/25), done.
✓ Repo prêt
best_model.pt  final_model.pt  probe_chembl251.pt  zinc_pretrained.pt


---
## Étape 1 — Pré-entraînement 1000 époques

> ⏱️ Cette cellule tourne ~8-10h. Ne pas fermer la session.

In [ ]:
import yaml, shutil

# Lire la config existante
with open('configs/pretrain_zinc.yaml') as f:
    config = yaml.safe_load(f)

# Modifier uniquement le nombre d'époques
config['entrainement']['num_epochs'] = 1000

# Sauvegarder une config dédiée 1000ep
with open('configs/pretrain_zinc_1000ep.yaml', 'w') as f:
    yaml.dump(config, f, allow_unicode=True)

print('✓ Config 1000ep créée')
print(f"  epochs : {config['entrainement']['num_epochs']}")
print(f"  lr     : {config['entrainement']['lr']}")
print(f"  batch  : {config['entrainement']['batch_size']}")

# Lancer le pré-entraînement
!python pretrain.py \
    --config configs/pretrain_zinc_1000ep.yaml \
    --epochs 1000

# Sauvegarde sur Drive
shutil.copy('checkpoints/zinc_pretrained.pt',
            f'{DRIVE_CKPT}/zinc_pretrained_1000ep.pt')
print('\n✓ Checkpoint 1000ep sauvegardé sur Drive')

✓ Config 1000ep créée
  epochs : 1000
  lr     : 0.0001
  batch  : 256
  Bio-JEPA — Pré-entraînement sur ZINC250k
  Protocole de transfert learning (Phase 1 / 2)

  Après ce script, lancez :
    python few_shot_eval.py \
        --checkpoint checkpoints/zinc_pretrained.pt \
        --target CHEMBL251 \
        --save-json results/few_shot_transfer.json

[Données] Chargement de ZINC250k...
  (Téléchargement automatique au premier appel, ~4 Mo)
[ZINC] Téléchargement de ZINC250k depuis GitHub...
  URL  : https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/master/models/zinc_properties/250k_rndm_zinc_drugs_clean_3.csv
  Dest : data/zinc/raw/250k_rndm_zinc_drugs_clean_3.csv
  ✓ Téléchargement réussi (22.6 Mo)
Processing...
[ZINC] 249,455 SMILES lus → conversion en graphes...
  50,000/249,455 traités...
  100,000/249,455 traités...
  150,000/249,455 traités...
  200,000/249,455 traités...
[ZINC] Conversion terminée : 249,455 graphes valides │ 0 SMILES invalides ignorés (0.0%)
[

---
## Étape 2 — Évaluation comparative : 100ep vs 1000ep

In [5]:
# Récupérer les résultats A2A 100ep depuis Drive (déjà calculés)
import shutil
shutil.copy(f'{DRIVE_RES}/few_shot_transfer.json', 'results/few_shot_100ep.json')
print('✓ Résultats 100ep récupérés depuis Drive (few_shot_transfer.json)')


✓ Résultats 100ep récupérés depuis Drive (few_shot_transfer.json)


In [6]:
# Évaluation du checkpoint 1000 époques
!python few_shot_eval.py \
    --checkpoint checkpoints/zinc_pretrained_1000ep.pt \
    --target CHEMBL251 \
    --n-values 10,50,100,200,500,1000 \
    --n-runs 5 \
    --save-json results/few_shot_1000ep.json

print('\n✓ Évaluation 1000ep terminée')

  Évaluation Few-Shot : Bio-JEPA vs GNN supervisé
  Cible : CHEMBL251 — A2A (adénosine)
  N ∈ [10, 50, 100, 200, 500, 1000]
  5 runs par N  │  Dispositif : cuda

[Données] Chargement du dataset ChEMBL — target=CHEMBL251
  Matérialisation du train set en liste...

  Train total : 6,739  │  Val : 842  │  Test : 843
  Checkpoint 'checkpoints/zinc_pretrained_1000ep.pt' absent → fallback : checkpoints/zinc_pretrained.pt

[Bio-JEPA] Checkpoint : checkpoints/zinc_pretrained.pt

[Bio-JEPA] Extraction des embeddings (Target Encoder figé)...
  Train : (6739, 256)  │  Val : (842, 256)  │  Test : (843, 256)  │  1.7s

──────────────────────────────────────────────────────────────────────
  Démarrage des expériences (6 valeurs de N × 5 runs)
  Bio-JEPA : 200 époques (sonde)  │  GNN sup. : 300 époques
──────────────────────────────────────────────────────────────────────

  ── N = 10 ──────────────────────────────────────────────
    Bio-JEPA  N=   10  run 1/5  →  r=-0.1300  ρ=-0.1286  RMSE=3.8379
  

In [7]:
import json, pandas as pd

with open('results/few_shot_100ep.json') as f:
    r100 = json.load(f)
with open('results/few_shot_1000ep.json') as f:
    r1000 = json.load(f)

N = r100['N_values']
rows = []
for i, n in enumerate(N):
    r100_val  = r100['bio_jepa'][i].get('mean_r') or r100['bio_jepa'][i].get('pearson_r', {}).get('mean', 0)
    r1000_val = r1000['bio_jepa'][i].get('mean_r') or r1000['bio_jepa'][i].get('pearson_r', {}).get('mean', 0)
    rows.append({'N': n, 'Bio-JEPA 100ep': round(r100_val, 3), 'Bio-JEPA 1000ep': round(r1000_val, 3),
                 'Δ (1000-100)': round(r1000_val - r100_val, 3)})

df = pd.DataFrame(rows)
print('=== COMPARAISON 100ep vs 1000ep — Pearson r ===')
print(df.to_string(index=False))

=== COMPARAISON 100ep vs 1000ep — Pearson r ===
   N  Bio-JEPA 100ep  Bio-JEPA 1000ep  Δ (1000-100)
  10           0.036            0.045         0.009
  50           0.130            0.130         0.000
 100           0.174            0.174        -0.000
 200           0.278            0.278         0.000
 500           0.465            0.464        -0.001
1000           0.542            0.540        -0.002


---
## Étape 3 — Sauvegarde Drive

In [9]:
for fname in ['few_shot_100ep.json', 'few_shot_1000ep.json']:
    shutil.copy(f'results/{fname}', f'{DRIVE_RES}/{fname}')

shutil.copy('checkpoints/zinc_pretrained_1000ep.pt',
            f'{DRIVE_CKPT}/zinc_pretrained_1000ep.pt')

print('✓ Résultats sauvegardés sur Drive')
print('✅ NOTEBOOK 1 TERMINÉ')

✓ Résultats sauvegardés sur Drive
✅ NOTEBOOK 1 TERMINÉ
